# 05 - Train YOLO11s (cloud training on Kaggle GPU)
Fine-tunes COCO-pretrained `yolo11s.pt` on the notebook 03 export (single class `car`, unlabelled area grayed out).
Settings come from `configs/train_yolo11s.yaml`, which is identical to `train_yolo11n.yaml` except the model, so the two detectors are comparable.

**Inputs:** notebook 03 output (`yolo_f0.1/` and `artifacts/splits/`). The MetaPKLot dataset is **not** needed here.
**Accelerator:** GPU T4 x2 (one GPU is used). Internet ON (installs `ultralytics==8.3.40`, the version on the Orin, and downloads the pretrained weights).

**How to run**
1. Interactive session, `SMOKE = True`: Run All (~3-5 min). Checks the whole pipeline with 1 epoch.
2. Set `SMOKE = False`, then **Save Version -> Save & Run All (Commit)**. Training runs in the background (no need to keep the tab open) and the outputs become attachable for notebook 06.

**Restarting:** if a commit stops early, attach that version's output and re-run: the notebook resumes from `last.pt`. If a finished record already exists it skips training.
Only the **val** split is used here. The test split stays untouched until notebook 06.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Install the pinned ultralytics (same version as the Orin so the .pt loads there). No torch upgrade.
import importlib.metadata as md_
try:
    have = md_.version("ultralytics")
except md_.PackageNotFoundError:
    have = None
if have != "8.3.40":
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.40"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    print(r.stdout[-2000:])
# torch >= 2.6 defaults torch.load(weights_only=True), which ultralytics 8.3.40 checkpoints do not satisfy.
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
# Training uses one GPU. Hiding the second one lets ultralytics use all 4 CPU cores for data loading
# (it divides the cores by the number of visible GPUs: 2 loader workers on "T4 x2"). Must be set before torch loads.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import torch, ultralytics
# Turn off experiment-tracker integrations (Kaggle's ray breaks the raytune callback; wandb can wait for a login).
# Runs before any trainer exists. If you saw a raytune error earlier, restart the kernel first.
from src.training import yolo_train as yt
print("tracker integrations:", yt.disable_integrations())
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# Settings
SMOKE = True                     # True: 1-epoch pipeline check. False: full training (use Save & Run All)
VARIANT = "yolo11s"
DEVICE = 0                       # first GPU
EXPORT_DIR = None                # set to the yolo_f0.1 folder path if auto-detection fails
EXTRA_ARGS = {}                  # optional ultralytics overrides; anything set here is recorded in args.yaml
PLOTS = True
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

import json, time
from pathlib import Path
from src.common.config import load_yaml, config_hash
from src.common.status import new_run_id, write_status
from src.training import yolo_train as yt
TCFG = load_yaml("configs/train_yolo11s.yaml")
DCFG = load_yaml("configs/dataset.yaml")
SUBSET_FRACTION = TCFG["subset_fraction"]
assert SUBSET_FRACTION == DCFG["dataset"]["subset_fraction"], "train and dataset configs disagree on subset_fraction"
EXPORT = Path(EXPORT_DIR) if EXPORT_DIR else yt.find_export_dir(INPUT_ROOT, SUBSET_FRACTION)
# Copy train+val to local disk: /kaggle/input is read-only and slow (~14 images/s), which made every val pass take ~3 min.
LOCAL = Path(WORK) / "yolo_local_f{}".format(SUBSET_FRACTION)
t0 = time.time(); print("staging to", LOCAL, yt.stage_export(EXPORT, LOCAL), "in", round(time.time() - t0, 1), "s")
DATA_YAML = yt.write_local_data_yaml(LOCAL, WORK + "/data/data_f{}.yaml".format(SUBSET_FRACTION), splits=("train", "val"))
infos = [p for p in yt.find_files(INPUT_ROOT, "split_info.json") if p.parent.name == "splits"]
SPLIT_INFO = json.loads(infos[0].read_text()) if infos else None
NAME = "{}_f{}{}".format(VARIANT, SUBSET_FRACTION, "_smoke" if SMOKE else "")
PROJECT = WORK + "/runs"
RUN_DIR = Path(PROJECT) / NAME
OUT = Path(WORK) / "artifacts" / "train" / NAME
OUT.mkdir(parents=True, exist_ok=True)
CFG_HASH = config_hash({"train": TCFG, "dataset": DCFG, "smoke": SMOKE, "extra": EXTRA_ARGS})
ARGS = yt.resolve_train_args(TCFG, DATA_YAML, PROJECT, NAME, smoke=SMOKE)
ARGS.update({"device": DEVICE, "plots": PLOTS}); ARGS.update(EXTRA_ARGS)
print("export:", EXPORT)
print("split manifest hash:", (SPLIT_INFO or {}).get("manifest_hash", "split_info.json NOT FOUND"))
for s in ("train", "val", "test"):
    print(" ", s, "images:", len(list((EXPORT / "images" / s).iterdir())))
print("run:", NAME, "| config hash:", CFG_HASH)
print("train args:", ARGS)

In [ ]:
# Restart logic: skip if a finished record exists, resume from last.pt if a partial run exists
import shutil
RECORD = OUT / "training_record.json"
SKIP_TRAIN, RESUME_FROM = False, None
if RECORD.exists() and (OUT / "best.pt").exists():
    SKIP_TRAIN = True
    print("finished record found, skipping training:", RECORD)
else:
    if not (RUN_DIR / "weights" / "last.pt").exists():
        prev = [p for p in yt.find_files(INPUT_ROOT, "last.pt") if p.parent.parent.name == NAME]
        if prev:  # output of an earlier (interrupted) version, attached as input
            shutil.copytree(str(prev[0].parent.parent), str(RUN_DIR), dirs_exist_ok=True)
            print("copied previous run from", prev[0].parent.parent)
    if (RUN_DIR / "weights" / "last.pt").exists():
        RESUME_FROM = RUN_DIR / "weights" / "last.pt"
        print("will resume from", RESUME_FROM)
    else:
        print("fresh training run")

In [ ]:
# Train
from ultralytics import YOLO
RUN_ID = new_run_id(VARIANT)
STATUS = OUT / "status.json"
duration = None; optimizer_name = None
if not SKIP_TRAIN:
    write_status(STATUS, "05_train_yolo11s", RUN_ID, "RUNNING", CFG_HASH, "training " + NAME)
    t0 = time.time()
    try:
        if RESUME_FROM:
            model = YOLO(str(RESUME_FROM))
            try:
                model.train(resume=True)
            except AssertionError as e:  # "nothing to resume": the earlier run had already finished
                print("resume:", e)
        else:
            model = YOLO(TCFG["model"])
            model.train(**ARGS)
    except Exception as e:
        write_status(STATUS, "05_train_yolo11s", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500])
        raise
    duration = time.time() - t0
    tr = getattr(model, "trainer", None)
    if tr is not None and getattr(tr, "optimizer", None) is not None:  # actual optimizer (optimizer=auto overrides lr0/momentum)
        pg = tr.optimizer.param_groups[0]
        optimizer_name = "{}(lr={}, momentum={})".format(type(tr.optimizer).__name__, pg.get("initial_lr", pg.get("lr")),
                                                   pg.get("betas", (pg.get("momentum"),))[0])
    print("training finished in {:.1f} min".format(duration / 60))
BEST = RUN_DIR / "weights" / "best.pt"
assert BEST.exists() or (OUT / "best.pt").exists(), "no best.pt produced"
print(sorted(p.name for p in (RUN_DIR / "weights").iterdir()) if (RUN_DIR / "weights").exists() else "weights in " + str(OUT))

In [ ]:
# Training curves (from results.csv: real logged values only)
import matplotlib.pyplot as plt
RES = RUN_DIR / "results.csv" if (RUN_DIR / "results.csv").exists() else OUT / "results.csv"
rows = yt.read_results(RES)
ep = [r["epoch"] for r in rows]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for k in ("train/box_loss", "train/cls_loss", "train/dfl_loss", "val/box_loss", "val/cls_loss", "val/dfl_loss"):
    if k in rows[0]:
        ax[0 if k.startswith("train") else 1].plot(ep, [r[k] for r in rows], label=k.split("/")[1])
for k in ("metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"):
    if k in rows[0]:
        ax[2].plot(ep, [r[k] for r in rows], label=k.split("/")[1])
for a, t in zip(ax, ("train loss", "val loss", "val metrics")):
    a.set_title(VARIANT + " " + t); a.set_xlabel("epoch"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(str(OUT / "training_curves.png"), dpi=120); plt.show()
print("epochs logged:", len(rows), "| best:", yt.best_epoch(rows))

In [ ]:
# Validate best.pt on the val split (test is not touched until notebook 06)
W = str(BEST if BEST.exists() else OUT / "best.pt")
vm = YOLO(W).val(data=str(DATA_YAML), split="val", imgsz=ARGS["imgsz"], batch=ARGS["batch"], device=DEVICE,
                 plots=PLOTS, project=str(OUT), name="val", exist_ok=True, verbose=False)
VAL = {"precision": float(vm.box.mp), "recall": float(vm.box.mr), "mAP50": float(vm.box.map50),
       "mAP50-95": float(vm.box.map), "speed_ms_per_image": {k: round(v, 3) for k, v in vm.speed.items()}}
print(json.dumps(VAL, indent=1))

In [ ]:
# Model stats and a short Kaggle GPU benchmark (labelled Kaggle; never used as Orin numbers)
import statistics
B = TCFG["benchmark"]
t0 = time.time(); bm = YOLO(W); load_s = time.time() - t0
stats = {}
try:
    from ultralytics.utils.torch_utils import get_flops, get_num_params
    stats = {"parameters": int(get_num_params(bm.model)), "gflops": round(float(get_flops(bm.model, ARGS["imgsz"])), 2)}
except Exception as e:
    print("parameter/FLOPs count failed (recorded as NOT MEASURED):", e)
imgs = sorted((LOCAL / "images" / "val").iterdir())
imgs = imgs[::max(1, len(imgs) // (B["n_images"] + B["warmup"]))][:B["n_images"] + B["warmup"]]
half = bool(B["half"]) and torch.cuda.is_available()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
infer_ms, total_ms = [], []
for i, p in enumerate(imgs):
    r = bm.predict(str(p), imgsz=ARGS["imgsz"], device=DEVICE, half=half, verbose=False)[0]
    if i >= B["warmup"]:
        infer_ms.append(r.speed["inference"]); total_ms.append(sum(r.speed.values()))
BENCH = {"environment": "Kaggle " + (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"),
         "precision": "fp16" if half else "fp32", "imgsz": ARGS["imgsz"], "batch": 1,
         "warmup_images": B["warmup"], "model_load_s": round(load_s, 3),
         "inference_ms": yt.latency_summary(infer_ms), "end_to_end_ms": yt.latency_summary(total_ms),
         "peak_gpu_memory_mb": round(torch.cuda.max_memory_allocated() / 2**20, 1) if torch.cuda.is_available() else yt.NOT_MEASURED}
print(json.dumps({"stats": stats, "bench": BENCH}, indent=1))

In [ ]:
# Save the training record (spec section 7), artifacts and status
import yaml
from src.common.hashing import sha256_file
from src.common.env_record import save_environment
for f in ("weights/best.pt", "weights/last.pt", "results.csv", "args.yaml", "results.png"):
    if (RUN_DIR / f).exists():
        shutil.copy2(str(RUN_DIR / f), str(OUT / Path(f).name))
used = yaml.safe_load(open(OUT / "args.yaml")) if (OUT / "args.yaml").exists() else {}
if duration is None or RESUME_FROM:   # resumed or skipped: prefer the cumulative time ultralytics logs
    if rows and "time" in rows[-1]:
        duration = rows[-1]["time"]
rec = yt.build_record(VARIANT, ARGS, used, rows, OUT / "best.pt", sha256_file(OUT / "best.pt"), SPLIT_INFO,
                      duration, optimizer_name, stats, BENCH, RUN_ID, CFG_HASH, SMOKE)
rec["resumed"] = bool(RESUME_FROM)
rec["training_duration_note"] = ("from results.csv; after a resume it may cover only the last session" if (RESUME_FROM or SKIP_TRAIN) else "wall time of this session")
rec["val_metrics_best_pt"] = VAL
rec["lr_schedule_note"] = "lr_schedule holds the configured values; with optimizer=auto ultralytics picks lr0 and momentum itself (see optimizer_used)"
rec["train_args_requested"] = {k: str(v) if isinstance(v, Path) else v for k, v in ARGS.items()}
RECORD.write_text(json.dumps(rec, indent=2, default=str))
save_environment(str(OUT / "environment.json"))
write_status(STATUS, "05_train_yolo11s", RUN_ID, "COMPLETE", CFG_HASH, "{} smoke={}".format(NAME, SMOKE),
             extra={"val": VAL, "best_epoch": rec["best_epoch"]})
print(json.dumps({k: rec[k] for k in ("model_variant", "smoke_test", "epochs_completed", "best_epoch", "optimizer_used",
                                     "training_duration_s", "parameters", "gflops_at_imgsz", "checkpoint_sha256",
                                     "dataset_split_hash")}, indent=1, default=str))
print("saved to", OUT, sorted(p.name for p in OUT.iterdir()))
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":  # committed run: keep the local copy out of the saved output
    shutil.rmtree(str(LOCAL), ignore_errors=True); print("removed staged copy", LOCAL)

## What to send back
Paste the output of the **Settings**, **Train** (last ~15 lines), **curves**, **val** and **record** cells, and the training-curves image if anything looks odd.
After the full run (`SMOKE = False`, Save & Run All), the version's output holds `artifacts/train/yolo11s_f0.1/` (best.pt, record, curves) and `runs/` (for resuming). Attach it to notebook 06.